## CAS4133 - Assignment 1 (Due 5/6 at 11:59 PM)

In this assignment, you will do continued pretraining a LLaMA-3.2-1B model on Korean Domain using [Unsloth](https://github.com/unslothai/unsloth?tab=readme-ov-file) with 4-bit quantization. 
<br>We use the Korean subset of the [Wikipedia dataset] to continually pretrain the model. The codebase structure is based on the official practice notebook provided by Unsloth.

---

### [Summary]

Your goal is to outperform a baseline model trained on a naïve Korean Wikipedia subset. You are encouraged to explore different ways to enhance performance.  
Some suggestions will be provided, but feel free to try other approaches as well. 


### [What to submit]

1. Your CAS4133-assn1.ipynb: Once you run the entire notebook with your desired solution, download the entire output as your own ipynb: 'File -> Download -> Download ipynb'.

2. A pdf report that include followings
    - **[Important]** HuggingFace Repository containing your final model. Format of ${User_Name}/{Repo_Name}
    - Describing what you’ve consider for improving performance
    - Describing what parts are modified/added compared to ipynb codebase
    - Comparing evaluation loss with naïvely trained model





### [Instructions]
Do Not Modify the followings

- Training Arguments in the commented section
- `max_steps` = 2500 (; lower values allowed for testing or shortened training)
- Test dataset
- Model architectures defined at the begining


Free to add custom functions or modify other components not mentioned above.

**The test dataset must never be used during training**


### [Grading: 10 pts total]

Evaluation will be based on the following two criteria:
<br>(1) Evaluation loss on test set
<br>(2) Downstream performance on private korean benchmark


- +4: if your model outperforms our baseline in (1)
- +4: if you implement something new (e.g., new tokenizer or filtering method)

- +1: if your model is top 70% in (2)
- +1: if your model is top 40% in (2)
- +1: if your model is top-3 in (2) — (; extra credit. +1 can be transferred to another assignment as bonus)



### [Notes]
- On the TA’s setup, training the naïve model takes approximately 5 hours.
- When uploading your model to HuggingFace Hub, 
    
    1. You must generate a write-access token before, and save token contents securely.
        For details, refer to the official [documentation](https://huggingface.co/docs/hub/security-tokens).
    
    2. Authentication is handled inside the provided notebook.  
       Run the login cell and enter your token when prompted.
---


## 0. Install Requirements

In [1]:
!pip install ipykernel==6.29.5
!pip install unsloth==2025.3.19
!pip install gdown==5.2.0
!pip install huggingface_hub==0.36.2
!pip uninstall torchao -y
!pip install transformers==4.51.3
!pip install peft==0.13.2
!pip install accelerate==0.34.0
!pip install torch==2.6.0+cu124 torchvision==0.21.0 torchaudio==2.6.0 --index-url https://download.pytorch.org/whl/cu124
!pip install xformers==0.0.29.post3 --index-url https://download.pytorch.org/whl/cu124
!pip install protobuf==3.20.3

INFO: pip is looking at multiple versions of unsloth-zoo to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of unsloth-zoo to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 11.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 10.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 11.5 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 11.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 10.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 11.6 MB/s

Note: Some dependency conflict warnings may appear above.
These are expected and do not affect functionality.

After running the installation cell above, you must **restart the kernel** before proceeding.

In [1]:
import os, random
import torch
import numpy as np

print(torch.cuda.is_available())
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Set seed
def set_seed(seed=1):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(1)

True


## 1. Load Pretrained Model

In [2]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048
dtype = None # None for auto detection. Bfloat16 for Ampere+ GPUs.
load_in_4bit = True # Use 4bit quantization to reduce memory usage.

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-1B-unsloth-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.3.19: Fast Llama patching. Transformers: 4.51.3.
   \\   /|    NVIDIA GeForce RTX 3090. Num GPUs = 1. Max memory: 23.684 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.6. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


<string>:30: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.


model.safetensors:   0%|          | 0.00/1.10G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/230 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/459 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

In [3]:
print(model.config)

LlamaConfig {
  "_attn_implementation_autoset": true,
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 128000,
  "eos_token_id": 128001,
  "head_dim": 64,
  "hidden_act": "silu",
  "hidden_size": 2048,
  "initializer_range": 0.02,
  "intermediate_size": 8192,
  "max_position_embeddings": 131072,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 16,
  "num_key_value_heads": 8,
  "pad_token_id": 128004,
  "pretraining_tp": 1,
  "quantization_config": {
    "bnb_4bit_compute_dtype": "bfloat16",
    "bnb_4bit_quant_type": "nf4",
    "bnb_4bit_use_double_quant": true,
    "llm_int8_enable_fp32_cpu_offload": false,
    "llm_int8_has_fp16_weight": false,
    "llm_int8_skip_modules": null,
    "llm_int8_threshold": 6.0,
    "load_in_4bit": true,
    "load_in_8bit": false,
    "quant_method": "bitsandbytes"
  },
  "rms_norm_eps": 1e-05,
  "rope_scaling": {
    "factor": 32.0,

Now, We add LoRA adapters so we only need to update 1 to 10% of all parameters.
<br>Also add `embed_tokens` and `lm_head` to allow the model to learn out of distribution data.

In [ ]:
# [R3] Korean tokenizer extension — must run BEFORE LoRA wraps embed_tokens/lm_head
EXP = os.environ.get("EXPERIMENT_MODE", "light")

if EXP.startswith("tokenizer"):
    print("[tokenizer] extending vocab with Korean syllables...")
    hangul_syllables = [chr(c) for c in range(0xAC00, 0xD7A4)]  # 가 ~ 힣
    existing_vocab = set(tokenizer.get_vocab().keys())
    new_hangul = [h for h in hangul_syllables if h not in existing_vocab]
    print(f"[tokenizer] hangul syllables not in LLAMA vocab: {len(new_hangul)} / {len(hangul_syllables)}")

    added = tokenizer.add_tokens(new_hangul)
    print(f"[tokenizer] added {added} tokens; new vocab size: {len(tokenizer)}")

    model.resize_token_embeddings(len(tokenizer))
    print(f"[tokenizer] resized model embeddings to {len(tokenizer)}")

    sample = "안녕하세요. 인공지능 기술이 빠르게 발전하고 있습니다."
    tokens = tokenizer.tokenize(sample)
    print(f"[tokenizer] sample: {sample}")
    print(f"[tokenizer] tokens ({len(tokens)}): {tokens}")
else:
    print(f"[tokenizer] skipped (mode={EXP})")


In [4]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 128,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",

                      "embed_tokens", "lm_head",], # Add for continual pretraining
    lora_alpha = 32,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    use_gradient_checkpointing = "unsloth", # Uses 30% less VRAM
    random_state = 3407,
    use_rslora = True,   # For rank stabilized LoRA
    loftq_config = None,
)

Unsloth: Offloading input_embeddings to disk to save VRAM
Unsloth: Offloading output_embeddings to disk to save VRAM


Unsloth 2025.3.19 patched 16 layers with 16 QKV layers, 16 O layers and 16 MLP layers.


Unsloth: Training embed_tokens in mixed precision to save VRAM
Unsloth: Training lm_head in mixed precision to save VRAM


## 2. Preparing Dataset

In [ ]:
import zipfile
from pathlib import Path

ZIP_PATH = os.environ.get("DATASET_ZIP", "ko_wiki_data.zip")

# 데이터가 어디든 풀려 있으면 스킵
if not any(Path(".").rglob("ko_wiki_train/dataset_info.json")):
    if not os.path.exists(ZIP_PATH):
        import gdown
        gdown.download(
            "https://drive.google.com/uc?id=1t4GZ_KkQqcNIxaZVsiz5tZEraNCpAwRv",
            ZIP_PATH,
        )
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall()

Downloading...
From (original): https://drive.google.com/uc?id=1t4GZ_KkQqcNIxaZVsiz5tZEraNCpAwRv
From (redirected): https://drive.google.com/uc?id=1t4GZ_KkQqcNIxaZVsiz5tZEraNCpAwRv&confirm=t&uuid=7ae6b507-0b7b-432d-b070-2df949528688
To: /ko_wiki_public.zip
100%|██████████| 139M/139M [00:12<00:00, 11.4MB/s] 


**[Note]** Remember to add the EOS_TOKEN to the tokenized output. Otherwise you'll get infinite generations.

In [6]:
# Wikipedia provides a title and an article text.
# Free to use any custom format you define
wikipedia_prompt = """위키피디아 기사
### 제목: {}

### 기사:
{}"""

EOS_TOKEN = tokenizer.eos_token # Must add EOS_TOKEN

def formatting_prompts_func(examples):
    titles = examples["title"]
    texts  = examples["text"]
    outputs = []
    
    for title, text in zip(titles, texts):
        # Must add EOS_TOKEN, otherwise your generation will go on forever!
        text = wikipedia_prompt.format(title, text) + EOS_TOKEN
        outputs.append(text)
    return { "text" : outputs, }

In [ ]:
from datasets import load_from_disk
from pathlib import Path

def find_split(name):
    for p in Path(".").rglob(f"{name}/dataset_info.json"):
        return str(p.parent)
    raise FileNotFoundError(name)

train_dataset = load_from_disk(find_split("ko_wiki_train"))
test_dataset  = load_from_disk(find_split("ko_wiki_public_test"))

train_dataset = train_dataset.map(formatting_prompts_func, batched=True)
test_dataset = test_dataset.map(formatting_prompts_func, batched=True)

print(f"Train size: {len(train_dataset)}")
print(f"Test size: {len(test_dataset)}")

Map:   0%|          | 0/80750 [00:00<?, ? examples/s]

Map:   0%|          | 0/2125 [00:00<?, ? examples/s]

Train size: 80750
Test size: 2125


In [8]:
print(train_dataset[1]['text'])

위키피디아 기사
### 제목: 우세르카프

### 기사:
우세르카프는 이집트 제5왕조의 첫 파라오이다. "그의 영혼은 강하다"라는 뜻이다. 베세르카프라는 별칭도 있다.

우세르카프는 제데프레의 손자로, 솁세스카프와 친척 관계였다. 솁세스카레가 죽은 후, 그의 누이이자 멘카우레의 딸인 켄트카웨스와 결혼하여 왕위를 이었다.

우세르카프는 제4왕조와 달리 사카라에 있는 조세르의 무덤 근처에 자신의 피라미드를 세웠다. 기존의 피라미드에서는 무덤 신전이 피라미드의 동쪽에 자리잡고 있었는데, 우세르카프는 이것을 남쪽으로 바꾸었다. 이것은 태양신 숭배신앙과 관련이 있는 것으로, 이후 제5왕조의 파라오들은 "태양 왕들"이라는 명칭으로 불린다.

이 신전의 뜰에서 발견된 우세르카프의 두상은 스핑크스를 제외하고는 이집트 고왕국 군주의 두상으로는 가장 커다란 것이다.

또한 우세르카프는 아부-구라브에 태양신전을 지었다. 이 곳에 원형적 오벨리스크를 세웠다. 이후 이 유적은 아크나톤이 아텐에게 바치는 봉헌물이 되었다. 이후 제5왕조의 파라오들은 이 옆에 자신들의 태양신전을 지었다.

참고 문헌 

 피터 클레이턴(Peter Clayton) 저, 정영목 역, 《파라오의 역사》, 까치, 2004

이집트 제5왕조의 파라오
기원전 25세기 파라오<|end_of_text|>


You are allowed to modify the train dataset composition.  
- For example, you may apply filtering or quality control to the current dataset.  
- Alternatively, you may include a different Korean dataset.  
- However, the test dataset used to evaluate pretraining perplexity must remain unchanged.


In [ ]:
import re
HANGUL_RE = re.compile(r"[가-힣]")
ARTICLE_MARKER = "### 기사:\n"
TAIL_KEYWORDS = ("참고 문헌", "참고 자료", "각주", "같이 보기", "외부 링크", "출처")

EXP = os.environ.get("EXPERIMENT_MODE", "light")
print(f"[Experiment] mode = {EXP}")

# 모드별 설정
PRESETS = {
    "baseline":         dict(min_len=0,    min_hangul=0.0, tail=False),
    "light":            dict(min_len=100,  min_hangul=0.3, tail=False),
    "tail":             dict(min_len=100,  min_hangul=0.3, tail=True),
    "heavy":            dict(min_len=500,  min_hangul=0.5, tail=False),
    "tail_heavy":       dict(min_len=500,  min_hangul=0.5, tail=True),
    "tokenizer":        dict(min_len=100,  min_hangul=0.3, tail=False),
    "tokenizer_tail":   dict(min_len=100,  min_hangul=0.3, tail=True),
    "tokenizer_heavy":  dict(min_len=500,  min_hangul=0.5, tail=False),
    "tokenizer_xheavy": dict(min_len=1000, min_hangul=0.6, tail=False),
}
cfg = PRESETS[EXP]
print(f"[Experiment] config = {cfg}")

def _body(text):
    idx = text.find(ARTICLE_MARKER)
    body = text[idx + len(ARTICLE_MARKER):] if idx >= 0 else text
    if EOS_TOKEN and body.endswith(EOS_TOKEN):
        body = body[:-len(EOS_TOKEN)]
    return body

def keep(ex):
    title = ex["title"]
    if title.endswith(("목록", "일람")) or "(동음이의)" in title:
        return False
    body = _body(ex["text"])
    if len(body) < cfg["min_len"]:
        return False
    if cfg["min_hangul"] > 0:
        if len(HANGUL_RE.findall(body)) / max(len(body), 1) < cfg["min_hangul"]:
            return False
    return True

def normalize(ex):
    text = ex["text"]
    idx = text.find(ARTICLE_MARKER)
    if idx < 0:
        return {"text": text}
    head_end = idx + len(ARTICLE_MARKER)
    head, body = text[:head_end], text[head_end:]

    eos = EOS_TOKEN if body.endswith(EOS_TOKEN) else ""
    if eos:
        body = body[:-len(eos)]

    if cfg["tail"]:
        for kw in TAIL_KEYWORDS:
            m = re.search(rf"\n\s*{re.escape(kw)}\s*\n", body)
            if m:
                body = body[:m.start()]
                break

    body = re.sub(r"\n{3,}", "\n\n", body)
    body = re.sub(r"[ \t]{2,}", " ", body)
    body = body.rstrip()
    return {"text": head + body + ("\n" + eos if eos else "")}

if EXP == "baseline":
    print(f"[{EXP}] no filtering, no normalization")
else:
    train_dataset = train_dataset.filter(keep).map(normalize)
    print(f"[{EXP}] Train size after filter: {len(train_dataset):,}")

You may also train a custom tokenizer.  
- The current setup uses a naive LLaMA tokenizer.  
- To improve your model’s performance on Korean text, you are free to train and use a better tokenizer.

## 3. Continued Pretraining

Training Configuration
- The training configuration is fixed to `batch_size = 32` and `max_steps = 2500`  
  (corresponding to approximately 80,000 training instances).  
- You may reduce `max_steps` for testing or shortened training purposes.

In [ ]:
from unsloth import is_bfloat16_supported
from unsloth import UnslothTrainer, UnslothTrainingArguments

TRAINER_PACKING = os.environ.get("TRAINER_PACKING", "false").lower() == "true"
TRAINER_GROUP_BY_LENGTH = os.environ.get("TRAINER_GROUP_BY_LENGTH", "false").lower() == "true"
print(f"[trainer] packing = {TRAINER_PACKING}, group_by_length = {TRAINER_GROUP_BY_LENGTH}")

trainer = UnslothTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = test_dataset, # Make sure to use original test_dataset for eval
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = TRAINER_PACKING,

    args = UnslothTrainingArguments(
        
        logging_steps = 50,
        output_dir = os.environ.get("OUTPUT_DIR", "outputs"),
        report_to = [],
        
        max_steps = int(os.environ.get("MAX_STEPS", 2500)), # DO NOT EXCEED 2500 steps for this assignment
        
        # Optional: You can remove these two lines if you do not need checkpointing.
        save_strategy = "no", 
        # save_steps = 500,
        
        group_by_length = TRAINER_GROUP_BY_LENGTH,
        
        ###### DO NOT CHANGE ######
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 8,
        warmup_steps = 10,
        warmup_ratio = 0.1,
        num_train_epochs = 1,

        learning_rate = 5e-5,
        embedding_learning_rate = 1e-5, # Select a 2 to 10x smaller learning rate for the embedding matrices
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,        
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        ###### DO NOT CHANGE ######
    
    ),
)

In [10]:
# Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = NVIDIA GeForce RTX 3090. Max memory = 23.684 GB.
2.367 GB of memory reserved.


Now runs the training code. It may take up to 5 hours to complete.

In [11]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 80,750 | Num Epochs = 1 | Total steps = 2,500
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 8 x 1) = 32
 "-____-"     Trainable parameters = 615,514,112/1,000,000,000 (61.55% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
50,2.362900
100,2.264600
150,2.253100
200,2.211300
250,2.223600
300,2.181900
350,2.188800
400,2.192700
450,2.176200
500,2.175000


Evaluate Trained Model and Print PPL

Naively Continued Pretrain model achieved 2.052 eval loss on test dataset.
<br>So your own trained model should have lower loss than 2.052

In [14]:
eval_results = trainer.evaluate()
eval_loss = eval_results["eval_loss"]

perplexity = torch.exp(torch.tensor(eval_loss))
print(f"Eval loss: {eval_loss:.3f}")
print(f"Perplexity: {perplexity:.2f}")

Eval loss: 2.052
Perplexity: 7.78


## 4. Upload Trained Model to Huggingface

## 5. Considerations for Improvement

If the desired performance of your model is not achieved, you can try following options:

1. Filtering on Pretraining Dataset for high-quality data
1. Consider Pretraining Dataset Mix(; KR - EN)
1. Train your own Korean Tokenizer
1. Etc.